# 🛰️ Subsistemas 04 & 05: Satellite Recon YOLOv8 & FBI Biometrics (`04-criminal-interstate` & `05-geospatial-recon`)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maykonlincolnusa/FRONTIER-OS-Frontier-Operating-System-/blob/main/notebooks/colab/04_Satellite_YOLOv8_xView_Recon_Colab_GPU.ipynb)

* **Agências Líderes:** US Army, National Geospatial-Intelligence Agency (NGA) & FBI
* **Anéis de Defesa:** `RING_2_FEDERAL` (Reconhecimento Orbital Sentinel-2 / xView) & `RING_3_INTERSTATE` (Biometria Facial ArcFace 512-d)
* **Objetivo:** Detectar pistas de pouso clandestinas no deserto/Amazônia, comboios de veículos de cartéis e realizar busca vetorial de procurados da lista **FBI Ten Most Wanted**.


In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FrontierSatelliteDetector(nn.Module):
    """Backbone CNN/C2f inspirada em YOLOv8 para classificação e detecção em tiles de satélite."""
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, num_classes),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

model = FrontierSatelliteDetector(num_classes=4).to(device).eval()
dummy_img = torch.randn(1, 3, 256, 256, device=device)

torch.onnx.export(
    model,
    dummy_img,
    "frontier_sat_yolov8.onnx",
    input_names=["satellite_rgb_tile"],
    output_names=["class_confidences"],
    dynamic_axes={"satellite_rgb_tile": {0: "batch"}, "class_confidences": {0: "batch"}},
    opset_version=17,
)
print("✅ Exportado com sucesso: frontier_sat_yolov8.onnx (Classes: Clandestine Airstrip, Narco Convoy, Go-Fast Vessel, Radar Site)")
